# RGBD Preprocessing — Single Sample Viewer

In [ ]:
# ── SET THESE THREE PATHS ─────────────────────────────────────────────────
BIN_PATH       = ""   # <sample>.bin           (point cloud / depth)
TEXTURE_PATH   = ""   # <sample>_texture.png   (raw RGB image)
SEGMENTED_PATH = ""   # <sample>.png           (red scanner segmentation → mask)
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, importlib
import numpy as np
import cv2
import matplotlib.pyplot as plt

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
UTIL_DIR = NOTEBOOK_DIR
if UTIL_DIR not in sys.path:
    sys.path.insert(0, UTIL_DIR)

import preprocessing.TumorDataset.tumor_dataset as tumor_dataset
importlib.reload(tumor_dataset)
from preprocessing.TumorDataset.tumor_dataset import Dataset
from preprocessing.preprocessing_functions._io import read_bin

In [ ]:
# ── Read the triplet ──────────────────────────────────────────────────────
rgb_img = cv2.imread(TEXTURE_PATH)        # _texture.png  -> RGB base image
seg_img = cv2.imread(SEGMENTED_PATH)      # .png          -> red scanner segmentation (mask source)

# Read the point cloud from the binary file
grid_x, grid_y, grid_z = read_bin(BIN_PATH)
depth_info = [(grid_x, grid_y, grid_z, os.path.basename(BIN_PATH))]

# The Dataset mixin methods operate on lists; wrap the single sample
rgb = [rgb_img]
seg = [seg_img]

# Instantiate Dataset as a method host
ds = Dataset(data_path=os.path.dirname(TEXTURE_PATH))

In [ ]:
# ── Full preprocessing pipeline (mirrors preprocess_images, read_bins=True) ─

og_masks  = [m.copy() for m in seg]
og_images = [im.copy() for im in rgb]

# --- RGB ---
images_rgb = ds.crop_raw_images(rgb)
masks      = ds.crop_masks(seg)
images_rgb, masks = ds.add_padding(images_rgb, masks)
masks      = ds.zoom_at(masks, 1.333, coord=None)
images_rgb = ds.crop_images(images_rgb)
masks      = ds.crop_images_offset(masks, x_offset=-25)
masks      = ds.create_binary_masks(masks)
masks      = ds.correct_binary_masks(masks)

# --- Depth (contour render) ---
masks_clone_depth = ds.crop_masks([m.copy() for m in og_masks])
images_depth_maps = ds.read_contours_array_depth(depth_info)
images_depth_maps = ds.crop_raw_images(images_depth_maps)
images_depth_maps, masks_clone_depth = ds.add_padding(images_depth_maps, masks_clone_depth)
images_depth_maps = ds.crop_images(images_depth_maps)

# --- RGD (depth infused into blue channel) ---
images_rgd = ds.read_contours_array_depth(depth_info)
images_rgd = ds.crop_raw_images(images_rgd)
images_rgd = ds.infuse_depth_into_blue_channel([im.copy() for im in og_images], images_rgd)
masks_clone_rgd = ds.crop_masks([m.copy() for m in og_masks])
images_rgd, masks_clone_rgd = ds.add_padding(images_rgd, masks_clone_rgd)
images_rgd = ds.crop_images(images_rgd)

# --- RGBD (4-channel sources) ---
images_rgbd_rgb    = images_rgb
images_rgbd_contor = images_depth_maps               # clone from depth maps
images_rgbd_grid   = ds.read_contours_with_grid(depth_info)

# shape report so a broken step is obvious
print("Done.")
print("  rgb        :", images_rgb[0].shape)
print("  mask       :", masks[0].shape)
print("  depth map  :", images_depth_maps[0].shape)
print("  rgd        :", images_rgd[0].shape)
print("  rgbd grid  :", images_rgbd_grid[0].shape)

In [ ]:
# ── Display ───────────────────────────────────────────────────────────────
panels = [
    (cv2.cvtColor(images_rgb[0], cv2.COLOR_BGR2RGB),         "RGB (texture)",        None),
    (masks[0],                                               "Binary mask",          "gray"),
    (images_depth_maps[0],                                   "Depth (contourf)",     "gray"),
    (cv2.cvtColor(images_rgd[0], cv2.COLOR_BGR2RGB),         "RGD (blue=depth)",     None),
    (images_rgbd_grid[0],                                    "RGBD-rawgrid",         "gray"),
]

fig, axes = plt.subplots(1, len(panels), figsize=(22, 4))
fig.suptitle(os.path.basename(TEXTURE_PATH), fontsize=12)

for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()